# Train the multimodal dementia model on Colab

**Workflow:** train here (GPU) → download `best_model.pt` → run inference locally.

First: **Runtime → Change runtime type → GPU (T4 is fine)**.

Sections: (1) get code, (2) install, (3) smoke test, (4a) mock OR (4b) real Kaggle data, (5) train, (6) download `.pt`.

## 1. Get the code
Clone your GitHub repo (edit the URL).

In [ ]:
REPO_URL = "https://github.com/<your-username>/Neuralyzer.git"
!git clone $REPO_URL
%cd Neuralyzer

## 2. Install dependencies
Colab already ships CUDA `torch`/`torchaudio`, so we install everything else (incl. Whisper + Kaggle CLI).

In [ ]:
!pip install -q transformers>=4.40.0 scikit-learn>=1.3.0 PyYAML>=6.0 tqdm>=4.65.0 soundfile>=0.12.1 openai-whisper kaggle

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 3. Smoke test (mock data)
Confirms the full pipeline runs and loss decreases. Downloads HuBERT + BERT on first run.

In [ ]:
!python scripts/smoke_test.py

## 4a. Quick option — mock data
Pipeline check only; **metrics are meaningless**. Skip to Section 5 to train on mock, or use 4b for real data.

## 4b. Real (open) data — Kaggle Pitt Cookie-Theft

English Cookie-Theft speech (Pitt Corpus re-upload). **Audio only**, so we generate transcripts with Whisper.

> **License note:** this re-upload is derived from DementiaBank (originally under a data-use agreement). Use for personal learning/reproduction only.

**Kaggle token:** on kaggle.com → Account → *Create New API Token* downloads `kaggle.json`. Upload it below.

In [ ]:
# Upload kaggle.json (from Kaggle > Account > Create New API Token)
from google.colab import files
import os, shutil
up = files.upload()  # choose kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed')

In [ ]:
# Download + unzip the dataset
!kaggle datasets download -d tahouramorovati/dementia-detection-using-speech -p /content/pitt --unzip

# Inspect the folder layout so we know the class sub-folder names
import os
for dp, dirs, files_ in os.walk('/content/pitt'):
    wavs = [f for f in files_ if f.lower().endswith(('.wav','.mp3','.flac'))]
    if wavs:
        print(dp, '->', len(wavs), 'audio files')

The loader auto-maps common folder names (`dementia/ad/cd → 1`, `control/hc/cn/cc → 0`). If your folders use different names, note them from the output above and pass a mapping via `--label-map` style config (see README).

**Generate transcripts once with Whisper** (GPU makes this quick; `base` is a good speed/quality trade-off):

In [ ]:
!python scripts/transcribe_whisper.py --data-root /content/pitt --model base --language en

## 5. Train

**Real data (full 5-seed protocol):**
```
!python train.py --config src/configs/default.yaml --dataset kaggle_pitt --data-root /content/pitt
```
**Mock (pipeline check):**
```
!python train.py --config src/configs/default.yaml --dataset mock --single-seed --max-epochs 3
```
OOM? Lower `train.batch_size` in the YAML or use `--max-epochs` for a quick test.

In [ ]:
!python train.py --config src/configs/default.yaml --dataset kaggle_pitt --data-root /content/pitt

## 6. Download the trained checkpoint
`train.py` writes `runs/best_model.pt` (weights + config bundled). Download it for local inference.

In [ ]:
from google.colab import files
import os
ckpt = 'runs/best_model.pt'
assert os.path.exists(ckpt), f'{ckpt} not found — did training finish?'
print('size (MB):', round(os.path.getsize(ckpt) / 1e6, 1))
files.download(ckpt)